# 🧠 HUẤN LUYỆN WORLD MODEL (TinyViT + CfC) CHO TAY ROBOT BIONIC
Notebook này mount Google Drive, preload dataset vào GPU RAM và huấn luyện inline.

**Yêu cầu:** Thư mục `le-wm/` phải được upload đầy đủ lên Drive tại `Bionic_Hand_LWM/le-wm/`.

### Quy trình:
1. Mount Drive
2. Cài đặt thư viện + Preload dataset 137MB vào GPU RAM
3. Huấn luyện inline (num_workers=0, augmentation GPU-native)

### Bước 1: Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Cài đặt & Preload GPU

In [ ]:
# ── 1. CÀI ĐẶT THƯ VIỆN ────────────────────────────────────────────────
import subprocess, sys
print("🔄 1. Cài đặt thư viện...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "stable-worldmodel[train]", "h5py", "hdf5plugin", "imageio", "ncps", "timm"],
    check=True
)
print("  ✓ Xong!\n")

# ── 2. IMPORT ─────────────────────────────────────────────────────────
import os, shutil, glob, threading, time
import h5py
import torch
from torch.utils.data import Dataset

# ── 3. ĐƯỜNG DẪN ──────────────────────────────────────────────────────
DRIVE_BASE      = "/content/drive/MyDrive/Bionic_Hand_LWM"
DRIVE_H5        = f"{DRIVE_BASE}/bionic_hand_dataset_v3_96.h5"
DRIVE_CKPT      = f"{DRIVE_BASE}/checkpoints/vit_cfc_weights.ckpt"
DRIVE_BACKUP    = f"{DRIVE_BASE}/checkpoints"
DRIVE_TRAIN_DIR = f"{DRIVE_BASE}/le-wm"

LOCAL_BASE     = "/content/dataset"
LOCAL_DATASETS = f"{LOCAL_BASE}/datasets"
LOCAL_H5       = f"{LOCAL_DATASETS}/bionic_hand_dataset_v3_96.h5"
LOCAL_CKPT_DIR = f"{LOCAL_BASE}/checkpoints"
LOCAL_CKPT     = f"{LOCAL_CKPT_DIR}/vit_cfc_weights.ckpt"

for d in [LOCAL_DATASETS, LOCAL_CKPT_DIR, DRIVE_BACKUP]:
    os.makedirs(d, exist_ok=True)

# ── 4. COPY DATASET ───────────────────────────────────────────────────
print("🔄 2. Copy dataset H5 từ Drive → SSD local...")
if not os.path.exists(DRIVE_H5):
    raise FileNotFoundError(f"❌ Không tìm thấy dataset tại: {DRIVE_H5}")
shutil.copy2(DRIVE_H5, LOCAL_H5)
print(f"  ✓ OK ({os.path.getsize(LOCAL_H5) / 1024 / 1024:.1f} MB)\n")

# ── 5. PRELOAD VÀO GPU RAM ────────────────────────────────────────────
print("🔄 3. Preload dataset vào GPU RAM...")
with h5py.File(LOCAL_H5, 'r') as f:
    pixels = torch.from_numpy(f['pixels'][:]).cuda()
    action = torch.from_numpy(f['action'][:]).cuda()
    ep_len = f['ep_len'][:]
    ep_offset = f['ep_offset'][:]

class GPUDataset(Dataset):
    def __init__(self, pixels, action, ep_len, ep_offset, transform=None):
        self.pixels = pixels
        self.action = action
        self.ep_len = ep_len
        self.ep_offset = ep_offset
        self.transform = transform
        self.length = len(pixels)
    
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        item = {'pixels': self.pixels[idx], 'action': self.action[idx]}
        if self.transform:
            item = self.transform(item)
        return item
    
    def get_dim(self, key):
        if key == 'action':
            return self.action.shape[1]
        raise ValueError(f"Unknown key: {key}")

gpu_dataset = GPUDataset(pixels, action, ep_len, ep_offset)
print(f"  ✓ {pixels.shape[0]} frames trên GPU ({pixels.nbytes / 1024 / 1024:.1f} MB VRAM)\n")

# ── 6. KIỂM TRA CHECKPOINT ────────────────────────────────────────────
print("🔄 4. Kiểm tra checkpoint để Resume...")
if os.path.exists(DRIVE_CKPT):
    shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
    print(f"  ✓ Tìm thấy checkpoint → RESUME")
else:
    print("  ℹ️  Không có checkpoint cũ → TRAIN MỚI từ Epoch 0")
print()


### Bước 3: Huấn luyện

In [ ]:
# ── TRAINING INLINE ───────────────────────────────────────────────────
import os
import sys

TRAIN_DIR = "/content/drive/MyDrive/Bionic_Hand_LWM/le-wm"
if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(f"❌ Không tìm thấy thư mục le-wm tại: {TRAIN_DIR}\n"
                            "   Kiểm tra lại đường dẫn Google Drive.")
os.chdir(TRAIN_DIR)
sys.path.insert(0, TRAIN_DIR)

import hydra
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf, open_dict
import lightning as pl
import stable_pretraining as spt
import stable_worldmodel as swm
from module import SIGReg
from utils import get_column_normalizer, get_img_preprocessor, SaveCkptCallback
import torch
from functools import partial
from pathlib import Path
from torchvision.transforms import v2

print(f"  ✓ Working directory: {os.getcwd()}")

# Load config với absolute path
config_dir = os.path.join(TRAIN_DIR, "config/train")
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="lewm", overrides=[
        "model=vit_tiny_cfc",
        "data=bionic_hand_v3_96",
        "trainer.max_epochs=100",
        "trainer.accelerator=gpu",
        "trainer.devices=1",
        "output_model_name=vit_cfc",
        "subdir=vit_cfc_exp",
        "wandb.enabled=False",
        "loader.num_workers=0",
        "loader.persistent_workers=False",
        "loader.prefetch_factor=null",
        "+trainer.enable_progress_bar=False",
    ])

# Sử dụng GPU dataset
dataset = gpu_dataset

# Augmentation GPU-native
aug_transform = v2.Compose([
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    v2.RandomResizedCrop(cfg.img_size, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    v2.RandomRotation(degrees=5),
])

with open_dict(cfg):
    cfg.model.action_encoder.input_dim = cfg.data.dataset.frameskip * dataset.get_dim("action")
    dataset.transform = aug_transform

# Split dataset
rnd_gen = torch.Generator().manual_seed(cfg.seed)
train_set, val_set = spt.data.random_split(
    dataset, lengths=[cfg.train_split, 1 - cfg.train_split], generator=rnd_gen
)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=cfg.loader.batch_size, shuffle=True, drop_last=True,
    generator=rnd_gen, num_workers=0
)
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=cfg.loader.batch_size, shuffle=False, drop_last=False, num_workers=0
)

# Model
world_model = hydra.utils.instantiate(cfg.model)

optimizers = {
    'model_opt': {
        "modules": 'model',
        "optimizer": dict(cfg.optimizer),
        "scheduler": {"type": "LinearWarmupCosineAnnealingLR"},
        "interval": "epoch",
    },
}

def lejepa_forward(self, batch, stage, cfg):
    ctx_len = cfg.history_size
    n_preds = cfg.num_preds
    lambd = cfg.loss.sigreg.weight
    batch["action"] = torch.nan_to_num(batch["action"], 0.0)
    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"]
    ctx_emb = emb[:, :ctx_len]
    ctx_act = act_emb[:, :ctx_len]
    tgt_emb = emb[:, n_preds:]
    pred_emb = self.model.predict(ctx_emb, ctx_act)
    output["pred_loss"] = (pred_emb - tgt_emb).pow(2).mean()
    output["sigreg_loss"] = self.sigreg(emb.transpose(0, 1))
    output["loss"] = output["pred_loss"] + lambd * output["sigreg_loss"]
    losses_dict = {f"{stage}/{k}": v.detach() for k, v in output.items() if "loss" in k}
    self.log_dict(losses_dict, on_step=True, sync_dist=True)
    return output

data_module = spt.data.DataModule(train=train_loader, val=val_loader)
world_model = spt.Module(
    model=world_model,
    sigreg=SIGReg(**cfg.loss.sigreg.kwargs),
    forward=partial(lejepa_forward, cfg=cfg),
    optim=optimizers,
)

# Trainer setup
run_id = cfg.get("subdir") or ""
run_dir = Path(swm.data.utils.get_cache_dir(sub_folder='checkpoints'), run_id)
run_dir.mkdir(parents=True, exist_ok=True)

object_dump_callback = SaveCkptCallback(
    run_name=cfg.output_model_name, cfg=cfg.model, run_dir=run_dir, epoch_interval=10,
)

trainer = pl.Trainer(
    **cfg.trainer,
    callbacks=[object_dump_callback],
    num_sanity_val_steps=1,
    logger=None,
    enable_checkpointing=True,
)

ckpt_path = run_dir / f"{cfg.output_model_name}_weights.ckpt"

# Auto-sync background
_stop_sync = threading.Event()
def _auto_sync(interval_sec=600):
    while not _stop_sync.wait(interval_sec):
        files = (
            glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.ckpt"), recursive=True) +
            glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.pt"), recursive=True)
        )
        if files:
            for f in files:
                shutil.copy2(f, DRIVE_BACKUP)
            print(f"\n  💾 [Auto-sync] Backed up {len(files)} file(s) → Drive")

_sync_thread = threading.Thread(target=_auto_sync, args=(600,), daemon=True)
_sync_thread.start()

print("🚀 Bắt đầu huấn luyện...")
manager = spt.Manager(
    trainer=trainer,
    module=world_model,
    data=data_module,
    ckpt_path=ckpt_path if ckpt_path.exists() else None,
)
manager()

# Final sync
_stop_sync.set()
print("\n Đồng bộ checkpoint cuối cùng lên Google Drive...")
final_files = (
    glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.ckpt"), recursive=True) +
    glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.pt"), recursive=True)
)
if final_files:
    for f in final_files:
        shutil.copy2(f, DRIVE_BACKUP)
        print(f"  ✓ {os.path.basename(f)} → Drive")
    print(f"\n🎉 HOÀN THÀNH! Đã backup {len(final_files)} file(s) lên Drive.")
else:
    print("  ⚠️ Không tìm thấy checkpoint nào.")
